## Open Tools, Open Sky
### Doing LSST Science with Public Rubin Data & LINCC Frameworks
#### Latest update: Jorge I. Zuluaga

## Template notebook

In [4]:
from util import *
import requests
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
from astroquery.jplsbdb import SBDB
import matplotlib.dates as mdates
from astropy.timeseries import LombScargle
import re
from scipy.signal import find_peaks

# Macros
DEG = np.pi / 180
RAD = 1 / DEG

# Directories
fig_dir='figures/'
data_dir='data/'
!mkdir -p $data_dir $fig_dir

# Reload util.py routines
%load_ext autoreload 
%autoreload 2

# Check that data files exists
get_data(data_dir = data_dir)

Data already exists: data/obs_sbn_X05_full.parquet
Data already exists: data/mpcorb_extended.dat.gz
Data already exists: data/asteroids-with-rotation.csv
Data already exists: data/X05_simplified.csv
Data already exists: data/obs_final_elements_geometry_H.parquet
Data already exists: data/sbdb_physical_properties.csv
Data already exists: data/mpc_designations.csv
Download complete or files already present.


## Build a catalogue

Each `.txt` file is a list of starred phase-curve figures, one filename per line:

`phase_allbands_<year>_<designation>.webp`

That name is the provisional designation with the space replaced by an underscore, so `phase_allbands_2006_ST360.webp` is the provid `2006 ST360`. The same asteroid can appear in more than one list. The catalogue keeps each provid once.

In [5]:
from pathlib import Path

FIGURE_PREFIX = "phase_allbands_"

def provid_from_figure_name(name):
    """Recover the provid from a starred figure filename.

    phase_allbands_2006_ST360.webp -> 2006 ST360
    """
    stem = Path(name.strip()).stem
    if not stem.startswith(FIGURE_PREFIX):
        raise ValueError(f"Unexpected figure name: {name!r}")
    year, designation = stem[len(FIGURE_PREFIX):].split("_", 1)
    return f"{year} {designation}"

def catalogue_from_starred_lists(files):
    """Build a unique provid catalogue from a list of starred .txt files."""
    seen = set()
    provids = []
    n_lines = 0
    for path in files:
        for line in Path(path).read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            n_lines += 1
            provid = provid_from_figure_name(line)
            if provid in seen:
                continue
            seen.add(provid)
            provids.append(provid)
    catalogue = (
        pd.DataFrame({"provid": provids})
        .sort_values("provid")
        .reset_index(drop=True)
    )
    return catalogue, n_lines

starred_files = sorted(Path(data_dir).glob("medellin_workshop_solarsystem-asteroid-starred-*.txt"))
catalogue, n_lines = catalogue_from_starred_lists(starred_files)

out = Path(data_dir) / "medellin-vro-catalogue.csv"
catalogue.to_csv(out, index=False)

print(f"{len(starred_files)} lists, {n_lines} names, {len(catalogue)} unique provids")
print(f"Wrote {out}")
catalogue.head()

7 lists, 611 names, 508 unique provids
Wrote data/medellin-vro-catalogue.csv


,provid
0,1978 UU1
1,1981 EY23
2,1986 PX5
3,1992 EP8
4,1997 JU3


## Overlap with the rotation catalogue

`asteroids-with-rotation.csv` identifies each object by its current `provid` and, when known, an earlier `alt_designation`. A catalogue asteroid is the same object if its provid matches either column.

In [ ]:
vro = pd.read_csv(data_dir + "medellin-vro-catalogue.csv")
rot = pd.read_csv(data_dir + "asteroids-with-rotation.csv")

same_provid = set(vro["provid"]) & set(rot["provid"])
alt_provid = set(rot["alt_designation"].dropna())
same_alt = (set(vro["provid"]) & alt_provid) - same_provid

n_vro = len(vro)
n_shared = len(same_provid) + len(same_alt)
print(f"{n_shared} of {n_vro} catalogue asteroids are in asteroids-with-rotation.csv")
print(f"  {len(same_provid)} match the current provid")
print(f"  {len(same_alt)} match an earlier designation: {', '.join(sorted(same_alt))}")

## Observations in df_set

`df_set` is the asteroid subset of `obs_sbn_X05_full.parquet`: a row whose `provid` matches `YYYY LL[NN]`. Every object in `asteroids-with-rotation.csv` has that form, so its observation count is the number of parquet rows with that `provid`. When the object also has an earlier designation, those rows are counted separately.

In [ ]:
obs_file = data_dir + "obs_sbn_X05_full.parquet"
rot = pd.read_csv(data_dir + "asteroids-with-rotation.csv")

ids = pd.concat([rot["provid"], rot["alt_designation"]]).dropna().astype(str).str.strip()
ids = sorted(set(ids) - {""})
counts = (
    pq.read_table(obs_file, columns=["provid"], filters=[("provid", "in", ids)])
    .to_pandas()["provid"]
    .value_counts()
)

obs_table = rot[["provid", "alt_designation"]].copy()
obs_table["n_obs"] = obs_table["provid"].map(counts).fillna(0).astype(int)
obs_table["n_obs_alt"] = obs_table["alt_designation"].map(counts).fillna(0).astype(int)
obs_table = obs_table.sort_values(["n_obs", "provid"]).reset_index(drop=True)

n_below = int((obs_table["n_obs"] < 100).sum())
print(f"{n_below} of {len(obs_table)} have fewer than 100 observations under the current provid")
print(
    f"n_obs ranges from {obs_table['n_obs'].min()} to {obs_table['n_obs'].max()} "
    f"(median {obs_table['n_obs'].median():.0f})"
)
obs_table

## Create the full catalogue

Start from `figures/figures_medellin/asteroid_analysis_results.csv`. That table is the Medellín run of the improved pipeline: the time span and the number of measurements in each filter, the voted rotation period, the joint multiband fit on the synthetic channel Lk, the touching-spheres and ellipsoid fits on Lk, the phase-corrected colors, and the solar-normalized reflectance.

Add the mean absolute magnitudes in `colors.parquet`, the SBDB physical properties (numbers without units), and the orbital elements in `obs_final_elements_geometry_H.parquet` at the epoch of each asteroid's first observation. The colors in the analysis table already come from the joint light-curve fit, so they are not taken from `colors.parquet`. A missing value stays NaN.

The ordered table is written at the repository root as `medellin-vroc-v<dd_mm_yyyy>.csv`, using the date of the run, and the same file is copied to `medellin-vroc-latest.csv`.

In [ ]:
from pathlib import Path

analysis_path = Path("figures/figures_medellin/asteroid_analysis_results.csv")
catalogue = pd.read_csv(analysis_path)
catalogue["provid"] = catalogue["provid"].astype(str).str.strip()
provids = catalogue["provid"].tolist()

color_columns = ["provid", "H_u", "H_g", "H_r", "H_i", "H_z", "H_y"]
colors = (
    pq.read_table(
        data_dir + "colors.parquet",
        columns=color_columns,
        filters=[("provid", "in", provids)],
    )
    .to_pandas()
    .drop_duplicates("provid")
)
catalogue = catalogue.merge(colors, on="provid", how="left")

def parse_quantity(value):
    """Number without its unit, or the original text when it is not a single number."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    match = re.fullmatch(
        r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)"
        r"(?:\s*(?:km3\s*/\s*s2|g\s*/\s*cm3|km|h))?",
        text,
        flags=re.IGNORECASE,
    )
    if match:
        return float(match.group(1))
    return text

sbdb = pd.read_csv(data_dir + "sbdb_physical_properties.csv", low_memory=False)
sbdb["provid"] = sbdb["provid"].astype(str).str.strip()
sbdb = sbdb[sbdb["provid"].isin(provids)].copy()
sbdb["value_clean"] = sbdb["value"].map(parse_quantity)
sbdb["sigma_clean"] = sbdb["sigma"].map(parse_quantity)

parameters = ["H", "diameter", "rot_per", "albedo", "GM", "density", "BV", "UB", "spec_B", "spec_T", "extent", "pole"]
values = (
    sbdb.pivot(index="provid", columns="parameter", values="value_clean")
    .reindex(columns=parameters)
    .rename(columns=lambda name: f"{name}_sbdb")
)
sigmas = (
    sbdb.pivot(index="provid", columns="parameter", values="sigma_clean")
    .reindex(columns=parameters)
    .rename(columns=lambda name: f"{name}_sbdb_err")
)
sbdb_wide = values.join(sigmas).reset_index()
sbdb_wide = sbdb_wide[["provid"] + [column for name in parameters for column in (f"{name}_sbdb", f"{name}_sbdb_err")]]
catalogue = catalogue.merge(sbdb_wide, on="provid", how="left")

designations = pd.read_csv(data_dir + "mpc_designations.csv")
designations["provid"] = designations["provid"].astype(str).str.strip()
designations = designations.drop_duplicates("provid")
names = catalogue[["provid"]].merge(designations, on="provid", how="left")
geometry = (
    pq.read_table(
        data_dir + "obs_final_elements_geometry_H.parquet",
        columns=["obstime", "mpc_designation", "a", "e", "q", "incl", "node", "peri", "M"],
        filters=[("mpc_designation", "in", names["mpc_designation"].dropna().unique().tolist())],
    )
    .to_pandas()
    .sort_values("obstime")
)
# After sorting by time, the first row of each asteroid is its first observation.
elements = (
    geometry.groupby("mpc_designation", as_index=False, sort=False)
    .first()
    .rename(columns={"obstime": "epoch_tdb"})
)
elements = names.merge(elements, on="mpc_designation", how="left")
catalogue = catalogue.merge(elements, on="provid", how="left")
catalogue["permid"] = catalogue["mpc_designation"].astype(str).str.extract(r"^\((\d+)\)", expand=False).astype("Int64")
catalogue = catalogue.sort_values("provid").reset_index(drop=True)

print(f"{len(catalogue)} asteroids, {catalogue.shape[1]} columns before ordering")
print(f"  mean H_r: {catalogue['H_r'].notna().sum()}")
print(f"  joint color g-r: {catalogue['g-r'].notna().sum()}")
print(f"  voted period: {catalogue['p_rot_dem_h'].notna().sum()}")
print(f"  Lk shape fit: {catalogue['chi_r_ell_k'].notna().sum()}")
print(f"  reflectance R_g: {catalogue['R_g'].notna().sum()}")
print(f"  SBDB H: {catalogue['H_sbdb'].notna().sum()}")
print(f"  orbital elements: {catalogue['a'].notna().sum()}")
catalogue.head()


In [ ]:
print_list(catalogue.columns)

### Catalogue columns

In [ ]:
"""
Columns:

Identification
provid, permid, mpc_designation,

Observations
epoch_utc_ini, epoch_utc_end,
n_obs, n_obs_u, n_obs_g, n_obs_r, n_obs_i, n_obs_z, n_obs_y,

Orbit
epoch_tdb, a, e, q, incl, node, peri, M,

Mean absolute magnitudes
H_u, H_g, H_r, H_i, H_z, H_y,

Phase-corrected colors
u-g, u-g_err, g-r, g-r_err, r-i, r-i_err, i-z, i-z_err, z-y, z-y_err, u-r, u-r_err,

Voted rotation
p_rot_dem_h, votes_dem, n_series_dem, bands_dem, periods_dem_h,

Multiband fit on Lk
p_rot_multi_used_h, p_rot_multi_h, f_rot_multi, k_harm_multi, bic,
amp_lsm_k, amp_photo_k, rms_multi, n_outliers_multi,

Shape on Lk: touching spheres
q_sph_k, q_sph_err_k, phi_sph_deg_k, phi_sph_err_deg_k, chi_r_sph_k,
amp_sph_k, m_mid_sph_k, m_sep_k, p_rot_sph_h_k,

Shape on Lk: ellipsoid
b_over_a_k, b_over_a_err_k, amp_ell_k, m_mid_ell_k, m_ab_k, p_rot_ell_h_k, chi_r_ell_k,

Reflectance
g-r_spec, g-r_spec_err, R_g, R_g_err, n_g_spec,
i-r_spec, i-r_spec_err, R_i, R_i_err, n_i_spec,
z-r_spec, z-r_spec_err, R_z, R_z_err, n_z_spec,
y-r_spec, y-r_spec_err, R_y, R_y_err, n_y_spec,

SBDB physical parameters
H_sbdb, H_sbdb_err, albedo_sbdb, albedo_sbdb_err,
diameter_sbdb, diameter_sbdb_err, extent_sbdb, extent_sbdb_err,
pole_sbdb, pole_sbdb_err, density_sbdb, density_sbdb_err,
rot_per_sbdb, rot_per_sbdb_err,
GM_sbdb, GM_sbdb_err, BV_sbdb, BV_sbdb_err, UB_sbdb, UB_sbdb_err,
spec_B_sbdb, spec_B_sbdb_err, spec_T_sbdb, spec_T_sbdb_err,

Dynamical
res_body, res_flag, res_type, res_kp, res_k, res_orbit
"""

In [ ]:
from datetime import date
from shutil import copy2

catalogue_columns = [
    # Identification
    "provid", "permid", "mpc_designation",
    # Observations
    "epoch_utc_ini", "epoch_utc_end",
    "n_obs", "n_obs_u", "n_obs_g", "n_obs_r", "n_obs_i", "n_obs_z", "n_obs_y",
    # Orbit
    "epoch_tdb", "a", "e", "q", "incl", "node", "peri", "M",
    # Mean absolute magnitudes
    "H_u", "H_g", "H_r", "H_i", "H_z", "H_y",
    # Phase-corrected colors from the joint light-curve fit
    "u-g", "u-g_err", "g-r", "g-r_err", "r-i", "r-i_err",
    "i-z", "i-z_err", "z-y", "z-y_err", "u-r", "u-r_err",
    # Voted rotation period
    "p_rot_dem_h", "votes_dem", "n_series_dem", "bands_dem", "periods_dem_h",
    # Multiband fit on the synthetic channel Lk
    "p_rot_multi_used_h", "p_rot_multi_h", "f_rot_multi", "k_harm_multi", "bic",
    "amp_lsm_k", "amp_photo_k", "rms_multi", "n_outliers_multi",
    # Touching spheres on Lk
    "q_sph_k", "q_sph_err_k", "phi_sph_deg_k", "phi_sph_err_deg_k", "chi_r_sph_k",
    "amp_sph_k", "m_mid_sph_k", "m_sep_k", "p_rot_sph_h_k",
    # Equator-on ellipsoid on Lk
    "b_over_a_k", "b_over_a_err_k", "amp_ell_k", "m_mid_ell_k", "m_ab_k",
    "p_rot_ell_h_k", "chi_r_ell_k",
    # Solar-normalized reflectance
    "g-r_spec", "g-r_spec_err", "R_g", "R_g_err", "n_g_spec",
    "i-r_spec", "i-r_spec_err", "R_i", "R_i_err", "n_i_spec",
    "z-r_spec", "z-r_spec_err", "R_z", "R_z_err", "n_z_spec",
    "y-r_spec", "y-r_spec_err", "R_y", "R_y_err", "n_y_spec",
    # SBDB physical parameters
    "H_sbdb", "H_sbdb_err", "albedo_sbdb", "albedo_sbdb_err",
    "diameter_sbdb", "diameter_sbdb_err", "extent_sbdb", "extent_sbdb_err",
    "pole_sbdb", "pole_sbdb_err", "density_sbdb", "density_sbdb_err",
    "rot_per_sbdb", "rot_per_sbdb_err",
    "GM_sbdb", "GM_sbdb_err", "BV_sbdb", "BV_sbdb_err", "UB_sbdb", "UB_sbdb_err",
    "spec_B_sbdb", "spec_B_sbdb_err", "spec_T_sbdb", "spec_T_sbdb_err",
    # Dynamical
    "res_body", "res_flag", "res_type", "res_kp", "res_k", "res_orbit",
]

for column in ["res_body", "res_flag", "res_type", "res_kp", "res_k", "res_orbit"]:
    if column not in catalogue.columns:
        catalogue[column] = np.nan

missing = [column for column in catalogue_columns if column not in catalogue.columns]
if missing:
    raise KeyError(f"Missing catalogue columns: {missing}")

catalogue = catalogue.reindex(columns=catalogue_columns)
root = Path("..").resolve()
stamp = date.today().strftime("%d_%m_%Y")
dated = root / f"medellin-vroc-v{stamp}.csv"
latest = root / "medellin-vroc-latest.csv"
catalogue.to_csv(dated, index=False)
copy2(dated, latest)
working = Path("medellin-vroc.csv")
catalogue.to_csv(working, index=False)
print(f"{len(catalogue)} asteroids, {catalogue.shape[1]} columns")
print(f"  {dated}")
print(f"  {latest}")
print(f"  {working.resolve()}")
print_list(catalogue.columns)

In [ ]:
catalogue.head()

In [ ]:
catalogue.loc[catalogue["provid"] == "2023 WK24", ["provid", "p_rot_dem_h", "votes_dem", "p_rot_multi_h", "chi_r_ell_k", "chi_r_sph_k"]]

## Column documentation

One row of the catalogue is one asteroid. A missing value is NaN. The ordered columns below are the ones written to `medellin-vroc-v<dd_mm_yyyy>.csv` and `medellin-vroc-latest.csv`.

Mean absolute magnitudes `H_u` … `H_y` come from `colors.parquet`. The observation counts, the voted period, the Lk fit, the shape parameters, the phase-corrected colors, and the reflectance come from `figures/figures_medellin/asteroid_analysis_results.csv`. Magnitudes in the rotation and shape blocks are absolute magnitudes on the synthetic channel Lk.

## Columns

### Identification

| Column | Unit | Meaning |
|---|---|---|
| `provid` | | Provisional MPC designation, `YYYY LLnn`. |
| `permid` | | Permanent number: the integer in parentheses in `mpc_designation`. NaN when the asteroid is not numbered. |
| `mpc_designation` | | MPC designation used for the geometry. Numbered objects look like `(4273) Dunhuang`; unnumbered ones keep the provisional designation. |

### Observations

The photometric time span and the number of measurements. These counts are the measurements in the Rubin data, one column per LSST filter.

| Column | Unit | Meaning |
|---|---|---|
| `epoch_utc_ini` | UTC | Time of the earliest measurement, `YYYY-MM-DD HH:MM:SS`. |
| `epoch_utc_end` | UTC | Time of the latest measurement. |
| `n_obs` | | Number of measurements in all filters. |
| `n_obs_u`, `n_obs_g`, `n_obs_r`, `n_obs_i`, `n_obs_z`, `n_obs_y` | | Number of measurements in that filter. |

### Orbit

Elements at `epoch_tdb`, taken from the geometry table at the first observation.

| Column | Unit | Meaning |
|---|---|---|
| `epoch_tdb` | TDB | Epoch of the elements, equal to the time of the first observation. |
| `a` | au | Semimajor axis. |
| `e` | | Eccentricity. |
| `q` | au | Perihelion distance. |
| `incl` | deg | Inclination to the ecliptic. |
| `node` | deg | Longitude of the ascending node. |
| `peri` | deg | Argument of perihelion. |
| `M` | deg | Mean anomaly. |

### Mean absolute magnitudes

Mean absolute magnitude in each LSST band, from `colors.parquet`. These means are not phase-corrected. The colors in the next section come from the joint light-curve fit, so `g-r` is not `H_g − H_r`.

| Column | Unit | Meaning |
|---|---|---|
| `H_u`, `H_g`, `H_r`, `H_i`, `H_z`, `H_y` | mag | Mean absolute magnitude in that LSST band. |

### Phase-corrected colors

Colors from the joint multiband fit. One rotation model is shared by the filters, so a color does not require the two filters on the same night. The reference filter is r. The u measurements are left out of the periodic model and compared with it afterwards. Each color has an `_err` column, the 1-sigma uncertainty from the fit.

| Column | Unit | Meaning |
|---|---|---|
| `u-g`, `g-r`, `r-i`, `i-z`, `z-y` | mag | Color, the difference of the two magnitude offsets. |
| `u-r` | mag | Magnitude offset of u relative to r. |

### Rotation: voted period

The period is chosen by a vote. The voters are the Lomb–Scargle maxima of the three filters with the most measurements, excluding u and y, plus a joint Lomb–Scargle of each consecutive pair and of all three together. Periods that agree to 3%, or whose photometric frequencies differ by one or two cycles per day, count as the same vote. A unanimous vote uses the joint period of all the filters in the vote. Otherwise the reported period is the Lomb–Scargle period of the filter with the most measurements among the series that cast the winning vote.

| Column | Unit | Meaning |
|---|---|---|
| `p_rot_dem_h` | h | Rotation period chosen by the vote. |
| `votes_dem` | | Number of series that voted for that period. |
| `n_series_dem` | | Number of series in the vote. |
| `bands_dem` | | Filters in the vote, for example `Lr+Lg+Li`. |
| `periods_dem_h` | h | Period of each series, written as `Lr=3.33,Lg=3.33,Lr+Lg=3.33`. |

### Rotation: multiband fit on Lk

The light curve is taken to be double-peaked, so the rotational period is twice the photometric period. The fit uses between 2 and 10 Fourier harmonics and keeps the order with the lowest Bayesian information criterion. `p_rot_multi_used_h` is the period that starts the fit, here the voted period. `p_rot_multi_h` is the period after the fit.

| Column | Unit | Meaning |
|---|---|---|
| `p_rot_multi_used_h` | h | Period used to start the multiband fit. |
| `p_rot_multi_h` | h | Rotational period from the fit, `24 / f_rot_multi`. |
| `f_rot_multi` | d⁻¹ | Rotational frequency. |
| `k_harm_multi` | | Number of Fourier harmonics in the adopted model. |
| `bic` | | Bayesian information criterion of that model. |
| `amp_lsm_k` | mag | Peak-to-peak amplitude of the harmonic model on Lk over one cycle. |
| `amp_photo_k` | mag | Width of the Lk magnitudes between the 1st and 99th percentiles, after clipping residuals at 3σ. |
| `rms_multi` | mag | Root-mean-square residual of the joint fit. |
| `n_outliers_multi` | | Number of measurements rejected by the clip. |

### Shape: touching spheres on Lk

Two spheres in contact, fitted to the folded Lk light curve. `q_sph_k` is reported with the smaller sphere second, so it is at most 1. The spin axis makes an angle `phi_sph_deg_k` with the line of sight: 90° is equator-on and 0° is pole-on.

| Column | Unit | Meaning |
|---|---|---|
| `q_sph_k` | | Radius ratio `r2/r1`. |
| `q_sph_err_k` | | Uncertainty of `q_sph_k`. |
| `phi_sph_deg_k` | deg | Angle between the spin axis and the line of sight. |
| `phi_sph_err_deg_k` | deg | Symmetric uncertainty of that angle. NaN when the fit only gives a lower limit. |
| `chi_r_sph_k` | | Reduced chi-squared of the fit. |
| `amp_sph_k` | mag | Peak-to-peak amplitude of the model. |
| `m_mid_sph_k` | mag | Midpoint of the model curve, halfway between its brightest and faintest magnitudes. |
| `m_sep_k` | mag | Magnitude when the two disks are fully separated. |
| `p_rot_sph_h_k` | h | Rotation period refined by the fit. |

### Shape: ellipsoid on Lk

Equator-on triaxial ellipsoid (`phi = 90°`) rotating about its short axis. The long axis is `a = 1`.

| Column | Unit | Meaning |
|---|---|---|
| `b_over_a_k` | | Axis ratio `b/a`. |
| `b_over_a_err_k` | | Uncertainty of `b/a`. |
| `amp_ell_k` | mag | Peak-to-peak amplitude of the model. |
| `m_mid_ell_k` | mag | Midpoint of the model curve. |
| `m_ab_k` | mag | Magnitude of the `a`–`b` face, the brightest aspect when seen equator-on. |
| `p_rot_ell_h_k` | h | Rotation period refined by the fit. |
| `chi_r_ell_k` | | Reduced chi-squared of the fit. |

### Reflectance

Solar-normalized reflectance from the joint light-curve colors, relative to r, where `R(r) = 1`. The solar colors subtracted from `m_b − m_r` are g−r = 0.45, i−r = −0.12, z−r = −0.16, and y−r = −0.18. The reflectance is `10^(−0.4 × excess)`.

For the g, i, z, and y filters, written below as `b`:

| Column | Unit | Meaning |
|---|---|---|
| `{b}-r_spec` | mag | Color `m_b − m_r`. For g this equals `g-r`. For i it equals `−(r−i)`. |
| `{b}-r_spec_err` | mag | Uncertainty of that color. |
| `R_b` | | Solar-normalized reflectance `R(b)/R(r)`. |
| `R_b_err` | | Uncertainty of the reflectance. |
| `n_b_spec` | | Number of measurements in filter `b`. |

The stored names are `g-r_spec`, `R_g`, `n_g_spec`, and the same pattern for i, z, and y.

### SBDB physical parameters

From the JPL Small-Body Database. The number is stored without its unit. Each quantity has an `_err` column; that uncertainty is NaN when SBDB does not give a numeric error. Taxonomic classes have no numeric error.

| Column | Unit | Meaning |
|---|---|---|
| `H_sbdb` | mag | Absolute magnitude. |
| `albedo_sbdb` | | Geometric albedo. |
| `diameter_sbdb` | km | Diameter. |
| `extent_sbdb` | km | Axis extents, kept as text `a x b` or `a x b x c`. |
| `pole_sbdb` | deg | Ecliptic pole, kept as text `longitude/latitude`. |
| `density_sbdb` | g cm⁻³ | Bulk density. |
| `rot_per_sbdb` | h | Rotation period in the literature. |
| `GM_sbdb` | km³ s⁻² | Gravitational parameter. |
| `BV_sbdb` | mag | Color B−V. |
| `UB_sbdb` | mag | Color U−B. |
| `spec_B_sbdb` | | Bus taxonomic class. |
| `spec_T_sbdb` | | Tholen taxonomic class. |

### Dynamics

Reserved for a mean-motion resonance. They are not computed yet, so every value is NaN.

| Column | Unit | Meaning |
|---|---|---|
| `res_body` | | Planet the asteroid is resonant with. |
| `res_flag` | | Whether a resonance is identified. |
| `res_type` | | Kind of resonance. |
| `res_kp` | | One integer of the resonance. |
| `res_k` | | The other integer of the resonance. |
| `res_orbit` | | Description of the resonant orbit. |
